# Лабораторная работа 4
## Хеширование
**(20 баллов)**
Выполните самостоятельно следующие задания и оформите отчет.

In [ ]:
class HashTable:
    def __init__(self):
        self.size = 11
        self.slots = [None] * self.size
        self.data = [None] * self.size

    def put(self, key, data):
        hashvalue = self.hashfunction(key, len(self.slots))

        if self.slots[hashvalue] == None:
            self.slots[hashvalue] = key
            self.data[hashvalue] = data
        else:
            if self.slots[hashvalue] == key:
                self.data[hashvalue] = data  # replace
            else:
                nextslot = self.rehash(hashvalue, len(self.slots))
                while self.slots[nextslot] != None and \
                        self.slots[nextslot] != key:
                    nextslot = self.rehash(nextslot, len(self.slots))

                if self.slots[nextslot] == None:
                    self.slots[nextslot] = key
                    self.data[nextslot] = data
                else:
                    self.data[nextslot] = data  # replace

    def hashfunction(self, key, size):
        return key % size

    def rehash(self, oldhash, size):
        return (oldhash + 1) % size

    def get(self, key):
        startslot = self.hashfunction(key, len(self.slots))

        data = None
        stop = False
        found = False
        position = startslot
        while self.slots[position] != None and \
                not found and not stop:
            if self.slots[position] == key:
                found = True
                data = self.data[position]
            else:
                position = self.rehash(position, len(self.slots))
                if position == startslot:
                    stop = True
        return data

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)

№ 1

**(5 балла)**

Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Реализуйте квадратичное пробирование как технику повторного хеширования.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы

In [6]:
class HashTable:
    def __init__(self):
        self.size = 11
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0

    def put(self, key, data):
        hashvalue = self.hashfunction(key, len(self.slots))
        
        if self.slots[hashvalue] is None:
            self.slots[hashvalue] = key
            self.data[hashvalue] = data
            self.count += 1
        else:
            if self.slots[hashvalue] == key:
                self.data[hashvalue] = data
            else:
                probe_count = 1
                nextslot = self.rehash(hashvalue, len(self.slots), probe_count)
                
                while (self.slots[nextslot] is not None and 
                       self.slots[nextslot] != key):
                    probe_count += 1
                    nextslot = self.rehash(hashvalue, len(self.slots), probe_count)
                    
                    if probe_count > self.size:
                        raise Exception("Хеш-таблица переполнена")

                if self.slots[nextslot] is None:
                    self.slots[nextslot] = key
                    self.data[nextslot] = data
                    self.count += 1
                else:
                    self.data[nextslot] = data

    def hashfunction(self, key, size):
        return key % size

    def rehash(self, oldhash, size, probe_count):
        return (oldhash + probe_count * probe_count) % size

    def get(self, key):
        startslot = self.hashfunction(key, len(self.slots))

        data = None
        stop = False
        found = False
        position = startslot
        probe_count = 0
        
        while (self.slots[position] is not None and 
               not found and not stop):
            
            if self.slots[position] == key:
                found = True
                data = self.data[position]
            else:
                probe_count += 1
                position = self.rehash(startslot, len(self.slots), probe_count)
                
                if position == startslot:
                    stop = True

                elif probe_count >= self.size:
                    stop = True

        return data

    def __getitem__(self, key):
        return self.get(key)

    def __setitem__(self, key, data):
        self.put(key, data)

    def __len__(self):
        return self.count

    def display(self):
        print("Slots:", self.slots)
        print("Data: ", self.data)

In [8]:
hash_table = HashTable()

hash_table[54] = 54
hash_table[26] = 26
hash_table[93] = 93
hash_table[40] = 40

print("После добавления первых значений:")
hash_table.display()
print(f"Текущий размер: {len(hash_table)}")

hash_table[15] = 15
print("\nПосле добавления 15 (коллизия с 26):")
hash_table.display()
print(f"Текущий размер: {len(hash_table)}")

print(f"\nПоиск ключа 15: {hash_table[15]}")
print(f"Поиск несуществующего ключа 29: {hash_table[29]}")

hash_table[15] = 150
print("\nПосле обновления значения по ключу 15:")
print(f"Поиск ключа 15: {hash_table[15]}")
print(f"Текущий размер: {len(hash_table)}")


После добавления первых значений:
Slots: [None, None, None, None, 26, 93, None, 40, None, None, 54]
Data:  [None, None, None, None, 26, 93, None, 40, None, None, 54]
Текущий размер: 4

После добавления 15 (коллизия с 26):
Slots: [None, None, None, None, 26, 93, None, 40, 15, None, 54]
Data:  [None, None, None, None, 26, 93, None, 40, 15, None, 54]
Текущий размер: 5

Поиск ключа 15: 15
Поиск несуществующего ключа 29: None

После обновления значения по ключу 15:
Поиск ключа 15: 150
Текущий размер: 5
